In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense


In [3]:
df1 = pd.read_csv("Normal_data.csv")
df2 = pd.read_csv("metasploitable-2.csv")
df3 = pd.read_csv("OVS.csv")

df = pd.concat([df1, df2, df3], axis=0)
df.reset_index(drop=True, inplace=True)

print("Merged Shape:", df.shape)


Merged Shape: (343889, 84)


In [4]:
df = df.drop_duplicates()
df = df.dropna()

print("After cleaning:", df.shape)

After cleaning: (343888, 84)


In [5]:
le = LabelEncoder()
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col])

In [8]:
X = df.drop("Label", axis=1)
y = df["Label"]

In [7]:
print(df.head())



   Flow ID  Src IP  Src Port  Dst IP  Dst Port  Protocol  Timestamp  \
0    33857   49931       443     344     53648         6        909   
1    33858   55004     53650     280       443         6        909   
2    65303   55004     35108     346        53         6        909   
3    65303   55006        53     344     35108         6        909   
4    15848   55004     60900     209       443         6        909   

   Flow Duration  Tot Fwd Pkts  Tot Bwd Pkts  ...  Fwd Seg Size Min  \
0         245230            44            40  ...                 0   
1        1605449           107           149  ...                 0   
2          53078             5             5  ...                 0   
3           6975             1             1  ...                 0   
4         190141            13            16  ...                 0   

   Active Mean  Active Std  Active Max  Active Min  Idle Mean  Idle Std  \
0          0.0         0.0         0.0         0.0        0.0       0.0

In [9]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)


In [10]:
pca = PCA(n_components=15)
X_pca = pca.fit_transform(X_scaled)

print("After PCA Shape:", X_pca.shape)


After PCA Shape: (343888, 15)


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X_pca, y, test_size=0.2, random_state=42, stratify=y
)


In [12]:
input_dim = X_train.shape[1]
input_layer = Input(shape=(input_dim,))

encoded = Dense(10, activation='relu')(input_layer)
encoded = Dense(8, activation='relu')(encoded)

decoded = Dense(10, activation='relu')(encoded)
decoded = Dense(input_dim, activation='sigmoid')(decoded)

autoencoder = Model(input_layer, decoded)
encoder = Model(input_layer, encoded)

autoencoder.compile(optimizer='adam', loss='mse')


In [13]:
autoencoder.fit(
    X_train, X_train,
    epochs=30,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)


Epoch 1/30
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 16s 3ms/step - loss: 0.1088 - val_loss: 0.0562
Epoch 2/30
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 22s 3ms/step - loss: 0.0554 - val_loss: 0.0545
Epoch 3/30
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 20s 3ms/step - loss: 0.0543 - val_loss: 0.0543
Epoch 4/30
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - loss: 0.0540 - val_loss: 0.0531
Epoch 5/30
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 20s 3ms/step - loss: 0.0531 - val_loss: 0.0530
Epoch 6/30
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - loss: 0.0529 - val_loss: 0.0529
Epoch 7/30
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - loss: 0.0527 - val_loss: 0.0526
Epoch 8/30
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 22s 3ms/step - loss: 0.0525 - val_loss: 0.0525
Epoch 9/30
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 20s 3ms/step - loss: 0.0524 - val_loss: 0.0525
Epoch 10/30
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 20s 3ms/step - loss: 0.0524 - val_loss: 0.0525
Epoch 11/30
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 21s 3ms/step - loss: 0.0524 - val_loss: 0.0524
Epoch 12/30
3869/38

In [14]:
X_train_enc = encoder.predict(X_train)
X_test_enc = encoder.predict(X_test)

print("Encoded Train Shape:", X_train_enc.shape)


8598/8598 ━━━━━━━━━━━━━━━━━━━━ 17s 2ms/step
2150/2150 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step
Encoded Train Shape: (275110, 8)


In [15]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train_enc, y_train)


RandomForestClassifier(n_estimators=200, random_state=42)

In [16]:
y_pred = rf.predict(X_test_enc)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.9998255256041176
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       281
           1       1.00      1.00      1.00        33
           2       1.00      1.00      1.00     14706
           3       1.00      1.00      1.00      9683
           4       1.00      1.00      1.00     10723
           5       1.00      1.00      1.00     13685
           6       1.00      1.00      1.00     19626
           7       0.75      1.00      0.86         3
           8       1.00      0.97      0.99        38

    accuracy                           1.00     68778
   macro avg       0.97      1.00      0.98     68778
weighted avg       1.00      1.00      1.00     68778

